# SaqBol — XLM-RoBERTa моделін оқыту

Бұл ноутбук алаяқтық SMS-ті анықтайтын трансформер-модельді (**XLM-RoBERTa**) оқытады және оны қазіргі TF-IDF моделімен **бірдей шартта** салыстырады.

**Іске қосу:**
1. `Runtime → Change runtime type → T4 GPU` таңдаңыз.
2. `Runtime → Run all` басыңыз.
3. Соңында Google Drive-қа рұқсат сұрайды — рұқсат беріңіз. Модель мен нәтижелер `MyDrive/saqbol/` папкасына сақталады.

**Не істейді:**
1. `train.csv` датасетін GitHub-тан жүктейді (2248 жол).
2. Датасетті шаблон бойынша 5 бөлікке бөледі.
3. Үш модельді салыстырады: қазіргі TF-IDF, TF-IDF + char n-gram, XLM-RoBERTa.
4. Нәтижені жалпы, тек нақты деректер, қазақша және орысша бойынша шығарады.
5. Соңғы модельді барлық деректе оқытып, сақтайды.

## 1. GPU тексеру

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "GPU жоқ! Runtime → Change runtime type → T4 GPU таңдап, ноутбукты қайта іске қосыңыз."
)
print("GPU:", torch.cuda.get_device_name(0))

## 2. Баптаулар

Әдетте ештеңе өзгертудің қажеті жоқ. Модель бір класты ғана болжай бастаса (F1 = 0), `LR`-ді `1e-5`-ке түсіріп көріңіз.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/DanialMyrzatayevMuratovich/saqbol/main/saqbol-ml/data/train.csv"

MODEL_NAME = "xlm-roberta-base"
MAX_LEN = 128        # SMS are short: the longest in train.csv is 144 characters
EPOCHS = 4
BATCH_SIZE = 16
LR = 2e-5
WARMUP_RATIO = 0.1
N_FOLDS = 5
SEED = 42

SAVE_TO_DRIVE = True
OUT_DIR = "saqbol-xlmr"
RESULTS_DIR = "results"

## 3. Датасетті жүктеу

`label == "scam"` → 1, қалғаны (`safe`, `suspicious`) → 0. Бұл сервистегі қазіргі модельмен бірдей.

Өз CSV файлыңызды қолданғыңыз келсе, төмендегі екі жолды ашыңыз (файлда кемінде `text`, `label` бағандары болуы керек).

In [ ]:
import json
import os
import random
import re
import shutil
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd

df = pd.read_csv(DATA_URL, keep_default_na=False)
# from google.colab import files
# df = pd.read_csv(next(iter(files.upload())), keep_default_na=False)

for column in ("lang", "source", "translated_from"):
    if column not in df:
        df[column] = ""
df["y"] = (df["label"] == "scam").astype(int)
df["is_real"] = ~df["source"].str.startswith("synthetic")

print(f"Барлығы: {len(df)} жол, оның нақты хабарламалар: {df.is_real.sum()}")
display(pd.crosstab(df["source"], df["label"], margins=True))
display(pd.crosstab(df["lang"].replace("", "?"), df["label"], margins=True))

## 4. Шаблон бойынша 5 бөлікке бөлу

Датасеттің ~90%-ы шаблоннан жасалған: көп жолдар бір-бірінен тек сандармен ерекшеленеді. Кездейсоқ бөлсек, «туыс» жолдар train мен test-ке екеуіне де түседі де, модель жаттап алған фразаны «таниды» — нәтиже жалған жоғары шығады.

Сондықтан сандарды `#` белгісіне ауыстырып, бір шаблоннан шыққан хабарламаларды **бір бөлікте** ұстаймыз. Бұл `train_baseline.py`-дағы әдіспен бірдей. Барлық модель **бірдей** бөліктерде бағаланады.

In [ ]:
from sklearn.model_selection import GroupKFold


def template_of(text: str) -> str:
    collapsed = re.sub(r"\d+", "#", text.lower())
    collapsed = re.sub(r"[^\w\s#]", "", collapsed)
    return re.sub(r"\s+", " ", collapsed).strip()


# An English row is a translation of another row. XLM-R is cross-lingual, so
# the pair must share a fold or the test fold leaks into training.
groups = df["translated_from"].where(df["translated_from"] != "", df["text"]).map(template_of)
df["fold"] = -1
for k, (_, test_idx) in enumerate(GroupKFold(n_splits=N_FOLDS).split(df["text"], df["y"], groups)):
    df.loc[df.index[test_idx], "fold"] = k

print(f"Шаблон саны: {groups.nunique()}")
display(pd.crosstab(df["fold"], df["label"]))

## 5. Метрикалар

Негізгі метрика — **scam класы бойынша F1 және recall** (ТЗ, NFR-2: recall ≥ 0.90). Нәтиже бес жиын бойынша шығады:
- `all` — барлық датасет;
- `real_only` — тек нақты хабарламалар (синтетикасыз). **Ең шынайы сан осы.**
- `kk`, `ru`, `en` — тіл бойынша (`en` — қазақша және орысша жолдардың аудармасы, нақты ағылшын SMS емес).

In [ ]:
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, roc_auc_score

SUBSETS = {
    "all": np.ones(len(df), dtype=bool),
    "real_only": df["is_real"].values,
    "kk": (df["lang"] == "kk").values,
    "ru": (df["lang"] == "ru").values,
    "en": (df["lang"] == "en").values,
}


def score(y, p, threshold=0.5):
    pred = (p >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y, pred, average="binary", zero_division=0
    )
    auc = roc_auc_score(y, p) if len(set(y)) > 1 else float("nan")
    return {
        "precision": round(float(precision), 4),
        "recall": round(float(recall), 4),
        "f1": round(float(f1), 4),
        "roc_auc": round(float(auc), 4),
    }


def report(results):
    rows = []
    y = df["y"].values
    for name, p in results.items():
        for subset, mask in SUBSETS.items():
            if mask.sum() == 0:
                continue
            rows.append({"model": name, "subset": subset, "n": int(mask.sum()), **score(y[mask], p[mask])})
    return pd.DataFrame(rows)


def to_markdown(table):
    header = "| " + " | ".join(table.columns) + " |"
    divider = "|" + "---|" * len(table.columns)
    body = ["| " + " | ".join(str(v) for v in row) + " |" for row in table.itertuples(index=False)]
    return "\n".join([header, divider, *body])


results = {}

## 6. Baseline: TF-IDF модельдері

- **TF-IDF word + LogReg** — қазір сервисте жұмыс істеп тұрған модель.
- **TF-IDF word+char + LogReg** — оған символдық n-gram қосылған нұсқа. Қазақ тілі жалғамалы: «картаңыз», «картаңызды», «картаға» — сөз деңгейінде үш бөлек сөз, ал символ деңгейінде ортақ «карта» бөлігі бар.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, Pipeline


def tfidf_word():
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])


def tfidf_word_char():
    features = FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True)),
    ])
    return Pipeline([
        ("tfidf", features),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ])


def out_of_fold(make_model):
    probs = np.zeros(len(df))
    for k in range(N_FOLDS):
        train, test = (df["fold"] != k).values, (df["fold"] == k).values
        model = make_model().fit(df["text"][train], df["y"][train])
        probs[test] = model.predict_proba(df["text"][test])[:, 1]
    return probs


results["TF-IDF word + LogReg"] = out_of_fold(tfidf_word)
results["TF-IDF word+char + LogReg"] = out_of_fold(tfidf_word_char)
display(report(results))

## 7. XLM-RoBERTa: оқыту функциялары

XLM-RoBERTa — 100 тілде (қазақ және орыс тілін қоса) алдын ала оқытылған трансформер. Біз оның үстіне «scam / not scam» классификаторын қосып, өз датасетімізде **fine-tune** жасаймыз.

In [ ]:
import gc

from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = device == "cuda"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def batches(texts, labels=None, shuffle=False):
    order = np.random.permutation(len(texts)) if shuffle else np.arange(len(texts))
    for start in range(0, len(order), BATCH_SIZE):
        chunk = order[start:start + BATCH_SIZE]
        batch = tokenizer(
            [texts[i] for i in chunk],
            truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt",
        )
        batch = {key: value.to(device) for key, value in batch.items()}
        if labels is not None:
            batch["labels"] = torch.tensor([labels[i] for i in chunk], device=device)
        yield batch


def train_transformer(texts, labels):
    set_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

    steps_per_epoch = (len(texts) + BATCH_SIZE - 1) // BATCH_SIZE
    total = steps_per_epoch * EPOCHS
    warmup = int(total * WARMUP_RATIO)

    def lr_lambda(step):
        # Linear warmup, then linear decay to zero.
        if step < warmup:
            return step / max(1, warmup)
        return max(0.0, (total - step) / max(1, total - warmup))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    model.train()
    for epoch in range(EPOCHS):
        running = 0.0
        for batch in batches(texts, labels, shuffle=True):
            optimizer.zero_grad()
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=USE_AMP):
                loss = model(**batch).loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running += loss.item()
        print(f"    epoch {epoch + 1}/{EPOCHS}: loss = {running / steps_per_epoch:.4f}")
    return model


@torch.no_grad()
def predict_proba(model, texts):
    model.eval()
    probs = []
    for batch in batches(texts):
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=USE_AMP):
            logits = model(**batch).logits
        probs.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(probs)


def free_memory():
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

## 8. XLM-RoBERTa: 5-fold cross-validation

Әр бөлік үшін модель қалған 4 бөлікте оқытылып, 5-ші бөлікте тексеріледі. T4-те бұл шамамен 10–20 минут алады. Әр эпохада `loss` азайып отыруы керек.

In [ ]:
texts = df["text"].tolist()
labels = df["y"].tolist()
probs = np.zeros(len(df))
started = time.time()

for k in range(N_FOLDS):
    fold_started = time.time()
    train_idx = np.where(df["fold"] != k)[0]
    test_idx = np.where(df["fold"] == k)[0]
    print(f"Fold {k + 1}/{N_FOLDS}: train={len(train_idx)}, test={len(test_idx)}")

    model = train_transformer([texts[i] for i in train_idx], [labels[i] for i in train_idx])
    probs[test_idx] = predict_proba(model, [texts[i] for i in test_idx])
    del model
    free_memory()

    fold_f1 = score(df["y"].values[test_idx], probs[test_idx])["f1"]
    print(f"    F1 = {fold_f1:.3f} ({time.time() - fold_started:.0f} с)")
    if fold_f1 == 0:
        print("    ⚠️ Модель бір класты ғана болжап тұр. LR = 1e-5 қойып, қайта іске қосыңыз.")

results["XLM-RoBERTa (fine-tune)"] = probs
print(f"Барлығы: {(time.time() - started) / 60:.1f} мин")

## 9. Нәтижелер — дипломға арналған кесте

`real_only` бағанына ерекше назар аударыңыз: бұл синтетикасыз, тек нақты хабарламалардағы сапа.

In [ ]:
table = report(results)
display(table)

print("\nF1 (scam):")
display(table.pivot(index="model", columns="subset", values="f1")[list(SUBSETS)])
print("Recall (scam):")
display(table.pivot(index="model", columns="subset", values="recall")[list(SUBSETS)])

best = "XLM-RoBERTa (fine-tune)"
print(f"\nConfusion matrix — {best} (жолдар: факт safe/scam, бағандар: болжам safe/scam):")
print(confusion_matrix(df["y"], (results[best] >= 0.5).astype(int)))

os.makedirs(RESULTS_DIR, exist_ok=True)
table.to_csv(f"{RESULTS_DIR}/metrics.csv", index=False)

with open(f"{RESULTS_DIR}/results.md", "w", encoding="utf-8") as handle:
    handle.write("# SaqBol — TF-IDF vs XLM-RoBERTa\n\n")
    handle.write(
        f"Датасет: {len(df)} жол ({int(df.is_real.sum())} нақты), "
        f"{N_FOLDS}-fold GroupKFold (шаблон бойынша). Метрикалар scam класы бойынша, шек 0.5.\n\n"
    )
    handle.write(to_markdown(table) + "\n")

oof = df[["text", "label", "lang", "source", "fold"]].copy()
for name, p in results.items():
    oof[name] = p.round(4)
oof.to_csv(f"{RESULTS_DIR}/oof_predictions.csv", index=False)

with open(f"{RESULTS_DIR}/metrics.json", "w", encoding="utf-8") as handle:
    json.dump({
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "samples": len(df),
        "real_samples": int(df.is_real.sum()),
        "templates": int(groups.nunique()),
        "folds": N_FOLDS,
        "config": {"model": MODEL_NAME, "max_len": MAX_LEN, "epochs": EPOCHS,
                   "batch_size": BATCH_SIZE, "lr": LR, "seed": SEED},
        "results": table.to_dict(orient="records"),
    }, handle, ensure_ascii=False, indent=2)

print(f"Нәтижелер {RESULTS_DIR}/ папкасына сақталды")

## 10. Қателерді талдау

Модель қай хабарламаларда қателесті? Бұл дипломның «Анализ ошибок» бөліміне керек.

In [ ]:
p = results["XLM-RoBERTa (fine-tune)"]
pd.set_option("display.max_colwidth", 120)

missed = df[(df["y"] == 1) & (p < 0.5)].assign(prob=p[(df["y"] == 1) & (p < 0.5)])
false_alarms = df[(df["y"] == 0) & (p >= 0.5)].assign(prob=p[(df["y"] == 0) & (p >= 0.5)])

print(f"Өткізіп алған скам (false negative): {len(missed)}")
display(missed.sort_values("prob")[["text", "lang", "source", "prob"]].head(15))
print(f"Жалған дабыл (false positive): {len(false_alarms)}")
display(false_alarms.sort_values("prob", ascending=False)[["text", "label", "lang", "source", "prob"]].head(15))

## 11. Соңғы модельді барлық деректе оқыту

Cross-validation тек бағалау үшін еді. Енді сервиске қоятын модельді **барлық** 2248 жолда оқытамыз және датасетте жоқ хабарламалармен тексереміз.

Соңғы төрт мысал «туысқан ақша сұрайды» схемасы — ол датасетте мүлдем жоқ. Модель оны ұстай ма, жоқ па, соны көреміз.

In [ ]:
final_model = train_transformer(texts, labels)
baseline = tfidf_word().fit(df["text"], df["y"])

examples = [
    "Служба безопасности банка, назовите код из смс",
    "Сәлеметсіз бе, банктің қауіпсіздік қызметі. Картаңыз бұғатталды, смс кодын айтыңыз",
    "Привет, встречаемся в 5",
    "Поздравляем с днём рождения!",
    "Сәлем, қайдасың? Кешке үйге келесің бе",
    "Ваш заказ доставлен в постамат, код получения 4821",
    "Мама, у меня проблемы, срочно скинь 20000 на этот номер",
    "Анашым, телефоным бұзылды, осы нөмірге 15000 жіберші тез",
    "Привет, это я, пишу с нового номера. Срочно нужны деньги, переведи 30000 на Kaspi",
    "Аға, WhatsApp-та дауыс беріп жіберші, сілтеме осы: vote-kz.site",
]
display(pd.DataFrame({
    "text": examples,
    "TF-IDF": baseline.predict_proba(examples)[:, 1].round(2),
    "XLM-R": predict_proba(final_model, examples).round(2),
}))

## 12. Сақтау

Модель (~1.1 GB) Google Drive-тағы `MyDrive/saqbol/` папкасына сақталады. `results/` папкасы (кестелер) жеке zip ретінде жүктеліп алынады.

In [ ]:
final_model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
with open(f"{OUT_DIR}/saqbol_meta.json", "w", encoding="utf-8") as handle:
    json.dump({
        "model_version": "xlmr-base-scam-v1",
        "base_model": MODEL_NAME,
        "trained_at": datetime.now(timezone.utc).isoformat(),
        "samples": len(df),
        "labels": {"0": "not_scam", "1": "scam"},
        "max_len": MAX_LEN,
    }, handle, ensure_ascii=False, indent=2)

shutil.make_archive("saqbol_results", "zip", RESULTS_DIR)

if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    target = "/content/drive/MyDrive/saqbol"
    shutil.copytree(OUT_DIR, f"{target}/{OUT_DIR}", dirs_exist_ok=True)
    shutil.copytree(RESULTS_DIR, f"{target}/{RESULTS_DIR}", dirs_exist_ok=True)
    print(f"Drive-қа сақталды: {target}")

try:
    from google.colab import files
    files.download("saqbol_results.zip")
except ImportError:
    pass

## Келесі қадам

1. `saqbol_results.zip` (немесе Drive-тағы `results/` папкасы) ішіндегі `results.md` пен `metrics.json`-ды жіберіңіз — дипломның эксперимент бөліміне кесте жасаймыз.
2. XLM-RoBERTa жақсы нәтиже көрсетсе, оны ML-сервиске қосамыз (ONNX-ке экспорттап, жылдамдатамыз).
3. Жаңа нақты деректер жиналған сайын `train.csv`-ті жаңартып, осы ноутбукты қайта іске қосасыз.